In [1]:
"""
Sepsis Prediction — LSTM v3
============================
Fixes vs v2:
  1. Unified data loading   → all folders combined into one pool
  2. Stratified split       → identical sepsis rate in train & val
  3. Feature normalisation  → StandardScaler before training
  4. Lower LR (1e-4)        → prevents epoch-1 peak then collapse
  5. MAX_TIMESTEPS = 48     → last 48 hours, removes noise from earlier days
  6. Scaler saved to disk   → required for inference on new patients
"""

import os
import csv
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
PARENT_DIR    = ".."
HEADER_FOLDER = os.path.join(PARENT_DIR, "processed_training_Ay")

# All folders that contain patient .psv files — add or remove as needed
ALL_FOLDERS = [
    os.path.join(PARENT_DIR, "processed_training"),
    os.path.join(PARENT_DIR, "processed_training_setB"),
    os.path.join(PARENT_DIR, "processed_training_Ay"),
]

SAVE_DIR      = "saved_models"
RESULTS_DIR   = "results"
MODEL_VERSION = "lstm_v3"

EARLY_HOURS   = 6    # predict sepsis this many hours before onset
MAX_TIMESTEPS = 48   # only use the last 48 hours — removes noise from earlier days
EPOCHS        = 50
BATCH_SIZE    = 64
PATIENCE      = 10
VAL_SIZE      = 0.2  # 80/20 stratified split
RANDOM_STATE  = 42

os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Device
# ─────────────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Feature columns
# ─────────────────────────────────────────────────────────────────────────────
all_columns = set()
if not os.path.exists(HEADER_FOLDER):
    raise FileNotFoundError(f"Header folder not found: {os.path.abspath(HEADER_FOLDER)}")

for fname in os.listdir(HEADER_FOLDER):
    if fname.endswith(".psv"):
        with open(os.path.join(HEADER_FOLDER, fname), "r") as fh:
            header = fh.readline().strip()
            if header:
                all_columns.update(header.split("|"))

FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")
N_FEATURES = len(FEATURE_COLUMNS)
print(f"Feature count: {N_FEATURES}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Load all patients from all folders into one unified pool
# ─────────────────────────────────────────────────────────────────────────────
def load_all_patients(folders, feature_columns, early_hours=6):
    """
    Load every .psv file from every folder into a single list.
    Sepsis patients: sequence ends `early_hours` before first sepsis label.
    Non-sepsis patients: full sequence.
    Duplicate files across folders are deduplicated by filename.
    """
    X, y = [], []
    seen_files = set()

    for folder in folders:
        if not os.path.exists(folder):
            print(f"  ⚠ Skipping missing folder: {folder}")
            continue
        for fname in sorted(os.listdir(folder)):
            if not fname.endswith(".psv"):
                continue
            if fname in seen_files:
                continue                        # skip duplicates across folders
            seen_files.add(fname)

            df = pd.read_csv(os.path.join(folder, fname), sep="|")
            df = df.reindex(columns=feature_columns + ["SepsisLabel"])
            df[feature_columns] = df[feature_columns].fillna(0.0)

            features = df[feature_columns].values.astype(np.float32)
            labels   = df["SepsisLabel"].values

            if labels.max() == 1:
                t_sepsis = np.where(labels == 1)[0][0]
                cutoff   = t_sepsis - early_hours
                if cutoff <= 0:
                    continue                    # not enough pre-sepsis data
                X.append(features[:cutoff])
                y.append(1)
            else:
                X.append(features)
                y.append(0)

    return X, np.array(y, dtype=np.int64)


print("\nLoading all patient data from all folders...")
X_all, y_all = load_all_patients(ALL_FOLDERS, FEATURE_COLUMNS, EARLY_HOURS)
print(f"Total patients : {len(y_all)}")
print(f"Sepsis (1)     : {y_all.sum()}  ({y_all.mean():.3f})")
print(f"No sepsis (0)  : {(y_all == 0).sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Stratified train / val split
#    Both splits will have the SAME sepsis rate — fixes the distribution mismatch
# ─────────────────────────────────────────────────────────────────────────────
indices = np.arange(len(y_all))
idx_train, idx_val = train_test_split(
    indices,
    test_size    = VAL_SIZE,
    stratify     = y_all,
    random_state = RANDOM_STATE,
)

X_train_raw = [X_all[i] for i in idx_train]
X_val_raw   = [X_all[i] for i in idx_val]
y_train     = y_all[idx_train]
y_val       = y_all[idx_val]

print(f"\nAfter stratified split:")
print(f"  Train : {len(y_train)} patients | sepsis rate: {y_train.mean():.4f}")
print(f"  Val   : {len(y_val)}  patients | sepsis rate: {y_val.mean():.4f}")

if abs(y_train.mean() - y_val.mean()) > 0.005:
    print("⚠ WARNING: sepsis rates differ — check stratify= argument")
else:
    print("✓ Sepsis rates match between train and val")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Pad / truncate to MAX_TIMESTEPS (last N hours)
# ─────────────────────────────────────────────────────────────────────────────
def pad_sequences(X, max_len):
    """Keep the LAST max_len timesteps; zero-pad shorter sequences at the start."""
    X_pad = np.zeros((len(X), max_len, N_FEATURES), dtype=np.float32)
    for i, seq in enumerate(X):
        L = len(seq)
        if L >= max_len:
            X_pad[i] = seq[-max_len:]
        else:
            X_pad[i, max_len - L:] = seq
    return X_pad

X_train = pad_sequences(X_train_raw, MAX_TIMESTEPS)
X_val   = pad_sequences(X_val_raw,   MAX_TIMESTEPS)
print(f"\nX_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. Feature normalisation (StandardScaler)
#    ICU features have very different scales (HR ~80, lactate ~2, WBC ~10000)
#    The LSTM cannot learn well without normalisation.
#    IMPORTANT: fit ONLY on train, apply to both — prevents data leakage.
# ─────────────────────────────────────────────────────────────────────────────
n_train, T, F = X_train.shape
n_val         = X_val.shape[0]

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, F)).reshape(n_train, T, F).astype(np.float32)
X_val   = scaler.transform(X_val.reshape(-1, F)).reshape(n_val, T, F).astype(np.float32)

scaler_path = os.path.join(SAVE_DIR, f"{MODEL_VERSION}_scaler.pkl")
joblib.dump(scaler, scaler_path)
print(f"\nScaler fitted and saved → {scaler_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Class imbalance weights
# ─────────────────────────────────────────────────────────────────────────────
class_counts    = np.bincount(y_train)
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\nClass counts — 0: {class_counts[0]}, 1: {class_counts[1]}")
print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")

sample_weights = np.where(y_train == 1,
                          1.0 / class_counts[1],
                          1.0 / class_counts[0])

pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32).to(device)

# ─────────────────────────────────────────────────────────────────────────────
# 8. DataLoaders
# ─────────────────────────────────────────────────────────────────────────────
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val, dtype=torch.float32),
)

sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float32),
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 9. Model — Bidirectional LSTM + Temporal Attention
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    """
    Bidirectional LSTM with temporal attention.

    - Bidirectional: reads sequence both forwards and backwards.
    - Attention: learns which of the 48 hours are most predictive
      instead of relying only on the final timestep.
    - No sigmoid in forward() — BCEWithLogitsLoss applies it internally
      which is numerically more stable.
    """
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,
        )
        lstm_out_dim = hidden_dim * 2           # *2 for bidirectional

        self.attention  = nn.Linear(lstm_out_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _  = self.lstm(x)                         # (B, T, H*2)
        attn_scores  = self.attention(lstm_out)              # (B, T, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)    # normalise over T
        context      = (lstm_out * attn_weights).sum(dim=1) # (B, H*2)
        return self.classifier(context)                      # (B, 1)

# ─────────────────────────────────────────────────────────────────────────────
# 10. Experiment logger
# ─────────────────────────────────────────────────────────────────────────────
LOG_PATH   = os.path.join(RESULTS_DIR, "model_log.csv")
LOG_FIELDS = ["timestamp", "version", "params", "epochs_run",
              "auroc", "auprc", "f1_sepsis", "recall_sepsis",
              "precision_sepsis", "threshold", "notes"]

def log_experiment(version, params, epochs_run, auroc, auprc,
                   f1, recall, precision, threshold, notes=""):
    row = {
        "timestamp":        datetime.now().strftime("%Y-%m-%d %H:%M"),
        "version":          version,
        "params":           str(params),
        "epochs_run":       epochs_run,
        "auroc":            round(auroc, 4),
        "auprc":            round(auprc, 4),
        "f1_sepsis":        round(f1, 4),
        "recall_sepsis":    round(recall, 4),
        "precision_sepsis": round(precision, 4),
        "threshold":        round(threshold, 3),
        "notes":            notes,
    }
    write_header = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  → Logged to {LOG_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 11. Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def get_val_probs(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            logits  = model(batch_x).squeeze(1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)


def best_threshold_by_f1(y_true, y_prob):
    """Find threshold that maximises F1 for the sepsis class using the PR curve."""
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_prob)
    denom  = precision_arr[:-1] + recall_arr[:-1]
    denom  = np.where(denom == 0, 1e-8, denom)
    f1_arr = 2 * (precision_arr[:-1] * recall_arr[:-1]) / denom
    best_idx = np.argmax(f1_arr)
    return thresholds[best_idx], f1_arr[best_idx]

# ─────────────────────────────────────────────────────────────────────────────
# 12. Training loop — early stopping + LR scheduler + gradient clipping
# ─────────────────────────────────────────────────────────────────────────────
def train_model(params, tag):
    print(f"\n{'='*55}")
    print(f"Training: {params}")
    print(f"{'='*55}")

    model = SepsisLSTM(
        input_dim  = N_FEATURES,
        hidden_dim = params["hidden_dim"],
        num_layers = params["num_layers"],
        dropout    = params["dropout"],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=params["lr"], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5
    )

    best_auroc       = 0.0
    best_state       = None
    patience_counter = 0
    history          = []

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x).squeeze(1)
            loss   = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        # ── Validate ───────────────────────────────────────────────────────
        y_prob_val = get_val_probs(model, val_loader)
        auroc      = roc_auc_score(y_val, y_prob_val)
        avg_loss   = total_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]

        scheduler.step(auroc)
        history.append({"epoch": epoch, "loss": avg_loss, "auroc": auroc})

        marker = ""
        if auroc > best_auroc:
            best_auroc       = auroc
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker           = " ✓ best"
            torch.save(best_state, os.path.join(SAVE_DIR, f"{tag}_best.pt"))
        else:
            patience_counter += 1

        print(f"  Epoch {epoch:02d}/{EPOCHS} | loss: {avg_loss:.4f} | "
              f"AUROC: {auroc:.4f} | lr: {current_lr:.2e}{marker}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, best_auroc, len(history)

# ─────────────────────────────────────────────────────────────────────────────
# 13. Grid search
# ─────────────────────────────────────────────────────────────────────────────
param_grid = ParameterGrid({
    "hidden_dim": [64, 128],
    "num_layers": [2],
    "dropout":    [0.3, 0.4],
    "lr":         [1e-4, 3e-4],   # lower LR than v2 — prevents epoch-1 collapse
})

best_overall_auroc  = 0.0
best_overall_model  = None
best_overall_params = None
best_overall_epochs = 0

for i, params in enumerate(param_grid):
    tag = f"{MODEL_VERSION}_run{i+1}"
    model, auroc, epochs_run = train_model(params, tag)
    if auroc > best_overall_auroc:
        best_overall_auroc  = auroc
        best_overall_model  = model
        best_overall_params = params
        best_overall_epochs = epochs_run

print(f"\n{'='*55}")
print(f"Best grid search AUROC : {best_overall_auroc:.4f}")
print(f"Best params            : {best_overall_params}")

# ─────────────────────────────────────────────────────────────────────────────
# 14. Final evaluation
# ─────────────────────────────────────────────────────────────────────────────
y_prob = get_val_probs(best_overall_model, val_loader)
auroc  = roc_auc_score(y_val, y_prob)
auprc  = average_precision_score(y_val, y_prob)
best_t, _ = best_threshold_by_f1(y_val, y_prob)
y_pred = (y_prob > best_t).astype(int)

report        = classification_report(y_val, y_pred, output_dict=True)
sepsis_key    = "1" if "1" in report else "1.0"
sepsis_metrics = report.get(sepsis_key, {})

print(f"\n{'='*55}")
print("FINAL PERFORMANCE REPORT")
print(f"{'='*55}")
print(f"ROC-AUC  : {auroc:.4f}   (target ≥ 0.80)")
print(f"AUPRC    : {auprc:.4f}   (target ≥ 0.30)")
print(f"Threshold: {best_t:.3f}  (target 0.3–0.5)")
print()
print(classification_report(y_val, y_pred))

# ─────────────────────────────────────────────────────────────────────────────
# 15. Save final model + log
# ─────────────────────────────────────────────────────────────────────────────
final_path = os.path.join(SAVE_DIR, f"{MODEL_VERSION}_final.pt")
torch.save(best_overall_model.state_dict(), final_path)
print(f"Model saved → {final_path}")

log_experiment(
    version    = MODEL_VERSION,
    params     = best_overall_params,
    epochs_run = best_overall_epochs,
    auroc      = auroc,
    auprc      = auprc,
    f1         = sepsis_metrics.get("f1-score", 0),
    recall     = sepsis_metrics.get("recall",   0),
    precision  = sepsis_metrics.get("precision",0),
    threshold  = best_t,
    notes      = (f"Stratified split, StandardScaler, 48-step window, "
                  f"bidirectional+attention, lr={best_overall_params['lr']}"),
)

# ─────────────────────────────────────────────────────────────────────────────
# 16. Architecture summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("Model architecture:")
print(f"{'='*55}")
print(best_overall_model)

print(f"\n{'='*55}")
print("Layer parameter shapes:")
print(f"{'='*55}")
for name, param in best_overall_model.named_parameters():
    if param.requires_grad:
        print(f"  {name:40} {tuple(param.shape)}")

# ─────────────────────────────────────────────────────────────────────────────
# 17. v1 → v2 → v3 comparison
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("Version comparison")
print(f"{'='*55}")
print(f"  {'Metric':<15} {'v1':>8} {'v2':>8} {'v3':>8}")
print(f"  {'-'*39}")
print(f"  {'AUROC':<15} {'0.617':>8} {'0.689':>8} {auroc:>8.3f}")
print(f"  {'F1 (sepsis)':<15} {'0.140':>8} {'0.335':>8} {sepsis_metrics.get('f1-score',0):>8.3f}")
print(f"  {'Recall':<15} {'0.270':>8} {'0.287':>8} {sepsis_metrics.get('recall',0):>8.3f}")
print(f"  {'Threshold':<15} {'0.900':>8} {'0.912':>8} {best_t:>8.3f}")

Using device: cuda
Feature count: 32

Loading all patient data from all folders...
Total patients : 39558
Sepsis (1)     : 2154  (0.054)
No sepsis (0)  : 37404

After stratified split:
  Train : 31646 patients | sepsis rate: 0.0544
  Val   : 7912  patients | sepsis rate: 0.0545
✓ Sepsis rates match between train and val

X_train : (31646, 48, 32)
X_val   : (7912, 48, 32)

Scaler fitted and saved → saved_models/lstm_v3_scaler.pkl

Class counts — 0: 29923, 1: 1723
Imbalance ratio : 17.4:1

Training: {'dropout': 0.3, 'hidden_dim': 64, 'lr': 0.0001, 'num_layers': 2}
  Epoch 01/50 | loss: 2.3081 | AUROC: 0.8152 | lr: 1.00e-04 ✓ best
  Epoch 02/50 | loss: 1.4877 | AUROC: 0.8294 | lr: 1.00e-04 ✓ best
  Epoch 03/50 | loss: 1.4267 | AUROC: 0.8360 | lr: 1.00e-04 ✓ best
  Epoch 04/50 | loss: 1.3810 | AUROC: 0.8405 | lr: 1.00e-04 ✓ best
  Epoch 05/50 | loss: 1.3412 | AUROC: 0.8473 | lr: 1.00e-04 ✓ best
  Epoch 06/50 | loss: 1.2866 | AUROC: 0.8549 | lr: 1.00e-04 ✓ best
  Epoch 07/50 | loss: 1.2570 